In [ ]:
import requests
api_key = 'ZZ7ZN51J8CYOT047'
keywords = 'IBM'
url = f'https://www.alphavantage.co/query?function=SYMBOL_SEARCH&keywords={keywords}&apikey={api_key}'
r = requests.get(url)
print(r.json())

{'bestMatches': [{'1. symbol': 'IBM', '2. name': 'International Business Machines Corp', '3. type': 'Equity', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '1.0000'}, {'1. symbol': 'IBMN', '2. name': 'ISHARES IBONDS DEC 2025 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8571'}, {'1. symbol': 'IBMO', '2. name': 'ISHARES IBONDS DEC 2026 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency': 'USD', '9. matchScore': '0.8571'}, {'1. symbol': 'IBMP', '2. name': 'ISHARES IBONDS DEC 2027 TERM MUNI BOND ETF ', '3. type': 'ETF', '4. region': 'United States', '5. marketOpen': '09:30', '6. marketClose': '16:00', '7. timezone': 'UTC-04', '8. currency':

In [33]:
import pandas as pd

In [ ]:
symbols = ['SWDA.LON']

for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )

    df = pd.read_csv(url, parse_dates=['timestamp'])
    df = df.sort_values('timestamp')  # oldest to newest

    # Use last 252 trading days for 1-year volatility
    df_last_year = df.tail(252)
    prices = df_last_year['close']
    
    # Convert GBX to GBP if needed
    prices_gbp = prices / 100.0

    # Calculate log returns
    returns = np.log(prices_gbp / prices_gbp.shift(1)).dropna()

    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GBP-based)")
    time.sleep(3)


Processing ERNS.LON...
ERNS.LON 1Y annualized volatility: 3.54% (GBP-based)


In [ ]:
symbols = ['SWDA.LON']

for symbol in symbols:
    print(f"Processing {symbol}...")

    url = (
        f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY'
        f'&symbol={symbol}&apikey={api_key}&outputsize=full&datatype=csv'
    )
    asset_df = pd.read_csv(url, parse_dates=['timestamp'])
    asset_df = df.sort_values('timestamp')  # oldest to newest
    # Use last 252 trading days for 1-year volatility
    df_last_year = asset_df.tail(252)


    fx_url = (
    f'https://www.alphavantage.co/query?function=FX_DAILY'
    f'&from_symbol=GBP&to_symbol=CHF&apikey={api_key}&outputsize=full&datatype=csv'
    )
    fx_df = pd.read_csv(fx_url, parse_dates=['timestamp'])
    fx_df = fx_df.sort_values('timestamp')
    fx_df = fx_df.set_index('timestamp')['close']  # Use 'close' as FX rate

    # Now merge with your asset price series on date and multiply:
    df = asset_df.join(fx_df, how='inner', rsuffix='_fx')
    df['close_chf'] = df['close'] * df['close_fx']  # Convert to CHF

    # Continue as before, but on 'close_chf'
    prices = df['close_chf']
    returns = np.log(prices / prices.shift(1)).dropna()


    # Daily volatility
    daily_vol = returns.std()

    # Annualized volatility
    annual_vol = daily_vol * np.sqrt(252)

    print(f"{symbol} 1Y annualized volatility: {annual_vol:.2%} (GBP-based)")
    time.sleep(3)


In [ ]:
import pandas as pd

fx_url = (
    f'https://www.alphavantage.co/query?function=FX_DAILY'
    f'&from_symbol=GBP&to_symbol=CHF&apikey={api_key}&outputsize=full&datatype=csv'
)
fx_df = pd.read_csv(fx_url, parse_dates=['timestamp'])
fx_df = fx_df.sort_values('timestamp')
fx_df = fx_df.set_index('timestamp')['close']  # Use 'close' as FX rate

# Now merge with your asset price series on date and multiply:
asset_df = ...  # Your asset prices DataFrame, index=timestamp, column='close'
df = asset_df.join(fx_df, how='inner', rsuffix='_fx')
df['close_chf'] = df['close'] * df['close_fx']  # Convert to CHF

# Continue as before, but on 'close_chf'
prices = df['close_chf']
returns = np.log(prices / prices.shift(1)).dropna()


